# Poster — Colab HTTP smoke test

Конечная проверка HTTP Engine без браузера и Drive credentials. Загрузите репозиторий с реализованным кодом или клонируйте опубликованную ветку. В Colab положите его в `/content/poster`, в Molab задайте свой `REPO`. Не предполагает бесконечную фоновую работу.

Установка, если пакет ещё не установлен: `python -m pip install -e /path/to/poster`. Playwright не нужен. Drive и перенос состояния описаны в `docs/google_drive_setup.md`; ключи задавайте через секреты среды.

In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

candidates = [Path.cwd(), Path("/content/poster"), Path("/workspace/poster")]
REPO = next((p for p in candidates if (p / "pyproject.toml").is_file()), None)
assert REPO is not None, "Upload/clone the implemented poster repository and set REPO first"
if importlib.util.find_spec("poster") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", str(REPO)], check=True)
print("Repository:", REPO)

In [ ]:
from poster.fixtures import serve_fixture
from poster.project import Project, load_project
from poster.runtime import Runtime
from poster.content_generator.generator import Generator

content = Generator(REPO / "examples/content").generate("italian_info", seed=42)
template = load_project(REPO / "examples/project_demo")
with serve_fixture() as (server, base):
    project = Project(
        template.site.model_copy(update={"base_url": base}), template.workflow, template.revision
    )
    result = await Runtime(project).run({"title": content.title, "body": content.body})
    assert result.status == "SUCCEEDED", result
    assert len(server.posts) == 1
    assert next(iter(server.posts.values()))["body"] == content.body
    print("HTTP + fresh CSRF + generated content:", result.status)
# Fixture is stopped; no background daemon or Chromium remains.

In [ ]:
import tempfile
from poster.persistence import State

with tempfile.TemporaryDirectory() as tmp:
    state = State(Path(tmp) / "state.sqlite")
    job = state.enqueue(project, {"title": content.title, "body": content.body}, "notebook-once", content)
    run_id = state.start(job)
    state.finish(job, run_id, result)
    state.backup(Path(tmp) / "snapshot.sqlite")
    State.restore(Path(tmp) / "snapshot.sqlite", Path(tmp) / "restored.sqlite")
    restored = State(Path(tmp) / "restored.sqlite")
    assert restored.rows()[0]["status"] == "SUCCEEDED"
    print("SQLite backup/restore:", restored.rows()[0]["status"])
    restored.close()
    state.close()

После выполнения fixture-сервер остановлен. Для настоящих проектов используйте отдельную локальную SQLite, согласованные backup и только один активный Engine. Этот notebook не проверяет политики времени жизни сервиса или реальный Google Drive.